In [2]:
# Cell 1: Install dependencies
!pip install timm huggingface_hub openslide-python h5py opencv-python-headless
!apt-get install -y openslide-tools

from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/hb-1968/prame-predict.git

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libopenslide0
Suggested packages:
  libtiff-tools
The following NEW packages will be installed:
  libopenslide0 openslide-tools
0 upgraded, 2 newly installed, 0 to remove and 42 not upgraded.
Need to get 104 kB of archives.
After this operation, 297 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libopenslide0 amd64 3.4.1+dfsg-5build1 [89.8 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 openslide-tools amd64 3.4.1+dfsg-5build1 [13.8 kB]
Fetched 104 kB in 1s (199 kB/s)
Selecting previously unselected package libopenslide0.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../libopenslide0_3.4.1+dfsg-5build1_amd64.deb ...
Unpacking libopenslide0 (3.4.1+dfsg-5build1) ...
Selecting previously unselected package openslide-tools.

In [6]:
# Cell 2: HuggingFace login
from huggingface_hub import login
login()

In [ ]:
import requests
import numpy as np
import pandas as pd
import torch
import timm
import h5py
import openslide
from pathlib import Path
from PIL import Image
from timm.data import resolve_data_config, create_transform
from tqdm import tqdm
from importlib.machinery import SourceFileLoader
from concurrent.futures import ThreadPoolExecutor

tile_module = SourceFileLoader("tile", "prame-predict/02_tile_wsi.py").load_module()

# ── Load model once ──
print("Loading UNI...")
model = timm.create_model(
    "vit_large_patch16_224",
    init_values=1e-5,
    num_classes=0,
    pretrained=False,
)

from huggingface_hub import hf_hub_download
ckpt_path = hf_hub_download(repo_id="MahmoodLab/uni", filename="pytorch_model.bin")
state_dict = torch.load(ckpt_path, map_location="cpu", weights_only=True)
model.load_state_dict(state_dict, strict=True)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Running on: {device}")

config = resolve_data_config(model.pretrained_cfg)
transform = create_transform(**config)

# ── Setup ──
manifest = pd.read_csv("prame-predict/data/expression/slide_manifest.csv")
drive_emb = Path("/content/drive/MyDrive/prame-predict/embeddings/uni")
drive_emb.mkdir(parents=True, exist_ok=True)
local_wsi = Path("/content/wsi_batch")
local_wsi.mkdir(exist_ok=True)

BATCH_SIZE_GPU = 64      # patches per forward pass
DOWNLOAD_BATCH = 75      # slides per download batch


def download_slide(row, max_retries=3):
    """Download a single slide from GDC with retry logic."""
    local_path = local_wsi / row["file_name"]
    if local_path.exists():
        return local_path

    for attempt in range(max_retries):
        try:
            url = f"https://api.gdc.cancer.gov/data/{row['file_id']}"
            response = requests.get(url, stream=True, timeout=300)
            response.raise_for_status()
            with open(local_path, "wb") as f:
                for chunk in response.iter_content(chunk_size=65536):
                    f.write(chunk)
            return local_path
        except Exception as e:
            print(f"  Retry {attempt + 1}/{max_retries} for {row['file_name']}: {e}")
            # Delete partial file
            if local_path.exists():
                local_path.unlink()

    print(f"  FAILED after {max_retries} attempts: {row['file_name']}")
    return None


def process_slide(slide_path, slide_name):
    """Tile one slide and extract features. Returns (features, coords) or None."""
    patches, coords = tile_module.tile_slide(slide_path)

    if len(patches) == 0:
        return None

    features = []
    with torch.no_grad():
        for j in range(0, len(patches), BATCH_SIZE_GPU):
            batch_patches = patches[j:j + BATCH_SIZE_GPU]
            tensors = torch.stack([transform(p) for p in batch_patches])
            feats = model(tensors.to(device))
            features.append(feats.cpu().numpy())

    return np.vstack(features), np.array(coords)


# ── Filter to unprocessed slides ──
remaining = []
for _, row in manifest.iterrows():
    emb_path = drive_emb / row["file_name"].replace(".svs", ".h5")
    if not emb_path.exists():
        remaining.append(row)

remaining = pd.DataFrame(remaining)
print(f"Slides remaining: {len(remaining)} / {len(manifest)}")

# ── Process in batches ──
for batch_start in range(0, len(remaining), DOWNLOAD_BATCH):
    batch = remaining.iloc[batch_start:batch_start + DOWNLOAD_BATCH]
    batch_end = min(batch_start + DOWNLOAD_BATCH, len(remaining))
    print(f"\n{'='*60}")
    print(f"BATCH {batch_start // DOWNLOAD_BATCH + 1}: "
          f"slides {batch_start + 1}–{batch_end} of {len(remaining)}")
    print(f"{'='*60}")

    # Download batch in parallel (8 threads)
    print(f"Downloading {len(batch)} slides...")
    with ThreadPoolExecutor(max_workers=8) as executor:
        futures = {
            executor.submit(download_slide, row): row["file_name"]
            for _, row in batch.iterrows()
        }
        for future in futures:
            try:
                future.result()
            except Exception as e:
                print(f"  Download error: {e}")
    print("Downloads complete")

    # Process each slide in the batch
    for _, row in batch.iterrows():
        slide_name = row["file_name"]
        local_path = local_wsi / slide_name
        emb_path = drive_emb / slide_name.replace(".svs", ".h5")

        print(f"\n  Processing {slide_name}...")

        if not local_path.exists():
            print(f"  Download failed, skipping")
            continue

        result = process_slide(local_path, slide_name)

        if result is None:
            print(f"  No patches, skipping")
            local_path.unlink()
            continue

        features, coords = result

        # Save to Drive
        with h5py.File(emb_path, "w") as f:
            f.create_dataset("features", data=features, compression="gzip")
            f.create_dataset("coords", data=coords)
            f.attrs["model"] = "uni"
            f.attrs["slide_name"] = slide_name
            f.attrs["num_patches"] = features.shape[0]
            f.attrs["feature_dim"] = features.shape[1]

        print(f"  Saved {features.shape} to Drive")

        # Delete processed slide
        local_path.unlink()

    # Clean up any remaining files in batch directory
    for f in local_wsi.glob("*.svs"):
        f.unlink()

    print(f"\nBatch complete, disk cleaned")

print(f"\nAll done! Embeddings saved to {drive_emb}")

Loading UNI...


pytorch_model.bin:   0%|          | 0.00/1.21G [00:00<?, ?B/s]

Running on: cuda
Slides remaining: 200 / 200

BATCH 1: slides 1–75 of 200
  Retry 1/3 for TCGA-EB-A4P0-01Z-00-DX1.9B525DEA-619A-43B6-B9C4-BB98511DEE56.svs: ('Connection broken: IncompleteRead(92969024 bytes read, 605598967 more expected)', IncompleteRead(92969024 bytes read, 605598967 more expected))
  Retry 1/3 for TCGA-D3-A3MV-06Z-00-DX1.C06EECC4-8DA7-4432-8FEC-7A5ED82E21CB.svs: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
